In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder

DATA_PATH = "synthetic_wsi_delhi_localities.csv"
df = pd.read_csv(DATA_PATH, parse_dates=["timestamp"])

TARGET = "safety_index"

# ── Time features ──
df["hour"] = df["timestamp"].dt.hour
df["day_of_week"] = df["timestamp"].dt.dayofweek
df["month"] = df["timestamp"].dt.month

# Cyclic encoding for hour & weekday
df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24)
df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24)
df["dow_sin"] = np.sin(2 * np.pi * df["day_of_week"] / 7)
df["dow_cos"] = np.cos(2 * np.pi * df["day_of_week"] / 7)

# ── NEW: Interaction features ──
df["lux_severity"] = df["lux_anomaly"] * df["severity_score"]  # how dangerous is darkness here

# ── Encode categorical features ──
cat_cols = ["crime_type", "victim_gender", "offender_gender", "locality", "location_type"]

label_encoders = {}
for col in cat_cols:
    le = LabelEncoder()
    df[col + "_enc"] = le.fit_transform(df[col].astype(str))
    label_encoders[col] = le

print(f"Encoded categorical columns: {cat_cols}")

# ── Drop unneeded columns ──
drop_cols = [
    "incident_id", "timestamp", "latitude", "longitude",
    "hour", "day_of_week", "month"  # we keep cyclic encodings instead
]
df_fe = df.drop(columns=drop_cols)

# Remove outliers
df_fe["safety_index"] = np.clip(df_fe["safety_index"], 0, 1)

# ── Define features & target ──
feature_cols = [c for c in df_fe.columns if c != TARGET]
X = df_fe[feature_cols]
y = df_fe[TARGET]

print(f"Feature count: {len(feature_cols)}")
print(f"X shape: {X.shape}, y shape: {y.shape}")

# Show all feature names
print(f"\nAll features:")
for i, col in enumerate(feature_cols):
    marker = " [NEW]" if col in [
        "location_type", "lux_reading", "expected_night_lux",
        "lux_anomaly", "shadow_penalty", "lux_x_night",
        "is_daytime", "location_type_enc", "lux_severity"
    ] else ""
    print(f"  {i+1:2d}. {col}{marker}")

# ── Save engineered dataset ──
OUT_PATH = "synthetic_wsi_delhi_engineered.csv"
df_fe.to_csv(OUT_PATH, index=False)
print(f"\nSaved engineered dataset: {OUT_PATH} (rows: {len(df_fe):,})")
